<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula7b_Classsifica%C3%A7%C3%A3o_Multiclasse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Carregar o dataset breast cancer.



In [28]:
from sklearn.datasets import load_wine

X, y = load_wine(return_X_y=True)

print(X.shape)
print(y.shape)
print(set(y))

(178, 13)
(178,)
{np.int64(0), np.int64(1), np.int64(2)}


In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (142, 13)
X_test shape: (36, 13)


In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit the scaler only on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data using the *same* fitted scaler
X_test_scaled = scaler.transform(X_test)

print("Data normalization complete.")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")
print("First 5 rows of X_train_scaled:")
print(X_train_scaled[:5])

Data normalization complete.
X_train_scaled shape: (142, 13)
X_test_scaled shape: (36, 13)
First 5 rows of X_train_scaled:
[[ 1.66529275 -0.60840587  1.21896194  1.60540017 -0.16738426  0.80400157
  -0.6916784   1.26722552  1.8775398   3.41947305 -1.65632857 -0.87940904
  -0.24860607]
 [-0.54952506  2.7515415   1.00331502  1.60540017 -0.30437887 -0.78538376
  -1.40123291  2.04959953 -0.87350523 -0.0248012  -0.58463272 -1.25462095
  -0.72992237]
 [-0.74531007 -1.14354109 -0.93750727 -0.28270426 -0.8523573   1.93702874
   1.7467906  -1.00165913  0.58798744 -0.24006834  0.35845962  0.2462267
  -0.24860607]
 [ 0.61294837 -0.61717858  1.00331502  0.87920616 -0.78385999  0.4892718
  -0.90154664  1.18898812  1.17258451  2.8813052  -1.65632857 -1.12955031
  -0.38138298]
 [ 0.11124931 -0.76631462 -0.93750727 -1.15413707 -0.16738426  0.17454204
   0.63748708 -0.68870952 -0.40926638 -0.58449577  0.95860929  0.1350528
   0.94638614]]


In [37]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
y_hot = encoder.fit_transform(y.reshape(-1, 1))
print(y_hot.shape)
idxs = [0, 70, 130]
print(y[idxs])
print(y_hot.toarray()[idxs])

(178, 3)
[0 1 2]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [45]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class OneHotEncoderCustom(BaseEstimator, TransformerMixin):
    def fit(self, y):
        self.unique_classes = np.unique(y)
        return self
    def transform(self, y):
        num_samples = len(y)
        num_classes = len(self.unique_classes)
        y_hot = np.zeros((num_samples, num_classes))
        for i, label in enumerate(y):
            class_idx = np.where(self.unique_classes == label)[0]
            y_hot[i, class_idx] = 1
        return y_hot

encoder = OneHotEncoderCustom()
y_hot = encoder.fit_transform(y)
print(y_hot.shape)
idxs = [0, 70, 5, 140, 100, 130]
print(y[idxs])
print(y_hot[idxs])

(178, 3)
[0 1 0 2 1 2]
[[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [77]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np

class LinearClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, learning_rate=0.01, num_iterations=1000):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.weights = None
        self.bias = None
        self.one_hot_encoder = OneHotEncoder()

    def fit(self, X, y):
        num_samples, num_features = X.shape
        num_classes = len(np.unique(y))
        y_hot = self.one_hot_encoder.fit_transform(y.reshape(-1, 1)).toarray()
        self.weights = np.zeros((num_features, num_classes))
        self.bias = 0
        for _ in range(self.num_iterations):
            logits = np.dot(X, self.weights) + self.bias
            y_predicted = logits > 0
            dw = np.dot(X.T, (y_predicted - y_hot))
            db = np.sum(y_predicted - y_hot)
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
        return self

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = np.argmax(linear_model, axis=1)
        return y_predicted.astype(int)

model = LinearClassifier()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 1.0


## 5. Comparar a performance da sua implementação com algum outro classificador linear do Scikit-Learn.

In [78]:
model = LinearClassifier()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0


In [80]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 1.0
